# 📁 PDF Batch Extractor - پردازش دسته‌ای PDF به Markdown

**Datalab SDK** - استخراج خودکار فایل‌های PDF با **پردازش همزمان ۵ فایل**

## ویژگی‌ها:
- 📁 انتخاب فایل یا پوشه با UI گرافیکی
- ⚡ **پردازش همزمان ۵ فایل با ۵ API مختلف** (افزایش سرعت ۵ برابری!)
- 🔄 **تلاش مجدد تا 5 بار در صورت خطا**
- 📊 نمایش Progress Bar و آمار لحظه‌ای
- 🖼️ استخراج خودکار تصاویر
- 💾 ذخیره خودکار در کنار هر PDF
- 📈 گزارش کامل در انتها با نمایش HTML زیبا
- 🗑️ امکان حذف خودکار فایل‌های تولید شده

---

## 🔧 Step 1: راه‌اندازی و انتخاب فایل‌ها

In [ ]:
# نصب پکیج‌های مورد نیاز
import subprocess, sys

_required_packages = {
    "nest_asyncio": "nest_asyncio",
    "ipywidgets": "ipywidgets",
    "PyPDF2": "PyPDF2",
    "datalab_sdk": "datalab-sdk",
    "openai": "openai",
}

print("📦 بررسی و نصب پکیج‌های مورد نیاز...")
for _import_name, _pip_name in _required_packages.items():
    try:
        __import__(_import_name)
    except ImportError:
        print(f"   ⬇️ نصب {_pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", _pip_name, "-q"])
print("✅ همه پکیج‌ها آماده‌اند!\n")

import nest_asyncio
nest_asyncio.apply()

# بارگذاری تمام تنظیمات و UI از ماژول خارجی
from pdf_setup import *

# نمایش وضعیت فایل‌های انتخاب شده
if pdf_files:
    print()
    print("="*80)
    print(f"✅ {len(pdf_files)} فایل آماده پردازش")
    total_size = sum(pdf.stat().st_size for pdf in pdf_files) / (1024 * 1024)
    print(f"💾 حجم کل: {total_size:.2f} MB")
    print(f"🚀 به Step 2 بروید برای شروع پردازش!")
    print("="*80)

## 🚀 Step 2: پردازش همزمان با ۵ Worker

**⚡ هر لحظه ۵ فایل به طور همزمان پردازش می‌شوند - سرعت ۵ برابری!**

**📝 توجه:** خروجی مستقیماً از Datalab API استخراج می‌شود (بدون پردازش اضافی GPT-5)

In [ ]:
# بارگذاری ماژول پردازش
from pdf_extractor_processor import process_pdfs
import os
import shutil
from pathlib import Path
from PyPDF2 import PdfReader, PdfWriter

async def split_and_process_pdfs(files):
    if not files:
        print("❌ هیچ فایلی انتخاب نشده است!")
        print("💡 لطفاً به Step 1 برگردید و فایل‌ها را انتخاب کنید")
        return

    chunks_map = {} # Map original file to list of chunk files
    all_files_to_process = []
    skipped_files = []  # فایل‌هایی که قبلاً پردازش شده‌اند
    
    # ایجاد پوشه موقت
    temp_dir = Path("temp_chunks_processing")
    if not temp_dir.exists():
        temp_dir.mkdir(exist_ok=True)

    print("✂️ در حال بررسی و تقسیم‌بندی فایل‌های بزرگ (محدودیت ۲۰ صفحه)...")
    
    for pdf_file in files:
        # 🔥 بررسی کش: آیا فایل md نهایی از قبل وجود دارد؟
        final_md_path = pdf_file.parent / f"{pdf_file.stem}.md"
        if final_md_path.exists():
            skipped_files.append(pdf_file)
            print(f"   💾 کش: {pdf_file.name} -> قبلاً پردازش شده، رد شد")
            continue
        
        try:
            reader = PdfReader(pdf_file)
            total_pages = len(reader.pages)
            
            if total_pages > 20:
                print(f"   📄 فایل {pdf_file.name} ({total_pages} صفحه) -> تقسیم به قطعات 20 صفحه‌ای")
                file_chunks = []
                
                for i in range(0, total_pages, 20):
                    writer = PdfWriter()
                    end_page = min(i + 20, total_pages)
                    for page_num in range(i, end_page):
                        writer.add_page(reader.pages[page_num])
                    
                    # نام‌گذاری قطعات: OriginalName_part_001.pdf
                    chunk_filename = f"{pdf_file.stem}_part_{i//20 + 1:03d}.pdf"
                    chunk_path = temp_dir / chunk_filename
                    
                    with open(chunk_path, "wb") as f:
                        writer.write(f)
                    
                    file_chunks.append(chunk_path)
                    all_files_to_process.append(chunk_path)
                
                chunks_map[pdf_file] = file_chunks
            else:
                all_files_to_process.append(pdf_file)
                
        except Exception as e:
            print(f"❌ خطا در خواندن {pdf_file.name}: {e}")
            all_files_to_process.append(pdf_file) # تلاش برای پردازش فایل اصلی در صورت خطا

    # نمایش آمار کش
    if skipped_files:
        print(f"\n💾 {len(skipped_files)} فایل از کش بارگذاری شد (قبلاً پردازش شده)")
    
    if not all_files_to_process:
        print("\n✅ همه فایل‌ها قبلاً پردازش شده‌اند! نیازی به پردازش مجدد نیست.")
        return {'success': [{'file': str(f), 'time': 0, 'pages': 0, 'images': 0, 'cached': True} for f in skipped_files], 'failed': []}
    
    print(f"✅ آماده پردازش {len(all_files_to_process)} فایل (شامل قطعات)")
    
    # اجرای پردازش روی همه فایل‌ها
    results = await process_pdfs(all_files_to_process)
    
    # اضافه کردن فایل‌های کش شده به نتایج
    for f in skipped_files:
        results['success'].append({'file': str(f), 'time': 0, 'pages': 0, 'images': 0, 'cached': True})
    
    # ساخت مجموعه فایل‌های ناموفق برای بررسی سریع
    failed_files_set = set(Path(item['file']).name for item in results['failed'])
    
    # ادغام نتایج
    print("\n🔄 در حال ادغام نتایج و جابجایی فایل‌ها...")
    
    for original_file, chunks in chunks_map.items():
        try:
            # 🔥 بررسی: آیا همه پارت‌ها موفق بودند؟
            all_chunks_successful = True
            failed_chunks = []
            
            for chunk in chunks:
                if chunk.name in failed_files_set:
                    all_chunks_successful = False
                    failed_chunks.append(chunk.name)
            
            # اگر حتی یک پارت ناموفق بود، فایل نهایی ساخته نشود
            if not all_chunks_successful:
                print(f"\n   ❌ فایل {original_file.name}: برخی پارت‌ها ناموفق بودند!")
                print(f"      پارت‌های ناموفق: {', '.join(failed_chunks)}")
                print(f"      ⚠️ فایل نهایی ساخته نمی‌شود - در حال حذف فایل‌های موقت...")
                
                # حذف فایل‌های md موفق پارت‌های این فایل
                for chunk in chunks:
                    # فایل md در temp_dir است
                    chunk_md_path = temp_dir / f"{chunk.stem}.md"
                    if chunk_md_path.exists():
                        try:
                            os.remove(chunk_md_path)
                            print(f"      🗑️ حذف شد: {chunk_md_path.name}")
                        except Exception as e:
                            print(f"      ⚠️ خطا در حذف {chunk_md_path.name}: {e}")
                    
                    # پوشه تصاویر در temp_dir است
                    chunk_images_dir = temp_dir / f"images_{chunk.stem}"
                    if chunk_images_dir.exists():
                        try:
                            shutil.rmtree(chunk_images_dir)
                            print(f"      🗑️ حذف شد: پوشه تصاویر {chunk.stem}")
                        except Exception as e:
                            print(f"      ⚠️ خطا در حذف پوشه تصاویر: {e}")
                
                # اضافه کردن فایل اصلی به لیست ناموفق‌ها (اگر قبلاً نیست)
                original_in_failed = any(Path(item['file']).name == original_file.name for item in results['failed'])
                if not original_in_failed:
                    results['failed'].append({
                        'file': str(original_file),
                        'error': f"پارت‌های ناموفق: {', '.join(failed_chunks)}"
                    })
                
                # حذف از لیست موفق‌ها اگر اشتباهی اضافه شده
                results['success'] = [item for item in results['success'] 
                                      if Path(item['file']).name != original_file.name]
                
                continue  # به فایل بعدی برو
            
            # همه پارت‌ها موفق بودند - ادغام
            merged_md_content = ""
            original_md_path = original_file.parent / f"{original_file.stem}.md"
            
            # ایجاد پوشه تصاویر نهایی واحد
            final_images_dir_name = f"images_{original_file.stem}"
            final_images_dir = original_file.parent / final_images_dir_name
            if not final_images_dir.exists():
                final_images_dir.mkdir(exist_ok=True)
                
            print(f"   🔗 ادغام {len(chunks)} قطعه برای {original_file.name}...")
            
            merge_successful = True
            for i, chunk in enumerate(chunks):
                try:
                    # فایل md در temp_dir است
                    chunk_md_path = temp_dir / f"{chunk.stem}.md"
                    
                    if chunk_md_path.exists():
                        # خواندن محتوای مارک‌داون
                        with open(chunk_md_path, "r", encoding="utf-8") as f:
                            content = f.read()
                        
                        # پوشه تصاویر در temp_dir است
                        chunk_images_dir = temp_dir / f"images_{chunk.stem}"
                        if chunk_images_dir.exists():
                            # پیشوند برای جلوگیری از تداخل نام تصاویر
                            img_prefix = f"part_{i+1:03d}_"
                            
                            for img_file in chunk_images_dir.iterdir():
                                if img_file.is_file():
                                    new_img_name = f"{img_prefix}{img_file.name}"
                                    target_img_path = final_images_dir / new_img_name
                                    
                                    # کپی/جابجایی فایل
                                    try:
                                        shutil.move(str(img_file), str(target_img_path))
                                    except Exception as img_err:
                                        print(f"      ⚠️ خطا در جابجایی تصویر {img_file.name}: {img_err}")
                                        # کپی به جای move
                                        try:
                                            shutil.copy2(str(img_file), str(target_img_path))
                                        except:
                                            pass
                                        
                            # بروزرسانی مسیرها در محتوای مارک‌داون
                            # جایگزینی مسیر قدیمی با مسیر جدید شامل پیشوند
                            old_path_prefix = f"images_{chunk.stem}/"
                            new_path_prefix = f"{final_images_dir_name}/{img_prefix}"
                            content = content.replace(old_path_prefix, new_path_prefix)
                            
                            # حذف پوشه تصاویر قطعه
                            try:
                                shutil.rmtree(chunk_images_dir)
                            except:
                                pass
                        
                        merged_md_content += content + "\n\n"
                        
                        # حذف فایل مارک‌داون موقت
                        try:
                            os.remove(chunk_md_path)
                        except:
                            pass
                    else:
                        print(f"      ⚠️ هشدار: خروجی برای {chunk.name} یافت نشد.")
                        merge_successful = False
                        
                except Exception as chunk_err:
                    print(f"      ❌ خطا در پردازش قطعه {chunk.name}: {chunk_err}")
                    merge_successful = False
            
            # فقط اگر ادغام موفق بود فایل نهایی ذخیره شود
            if merge_successful and merged_md_content.strip():
                with open(original_md_path, "w", encoding="utf-8") as f:
                    f.write(merged_md_content)
                print(f"      ✅ فایل نهایی ذخیره شد: {original_md_path.name}")
            else:
                print(f"      ❌ ادغام ناموفق برای {original_file.name} - فایل نهایی ذخیره نشد")
                # حذف پوشه تصاویر نهایی اگر خالی یا ناقص است
                if final_images_dir.exists():
                    try:
                        shutil.rmtree(final_images_dir)
                    except:
                        pass
                # اضافه کردن به لیست ناموفق‌ها
                results['failed'].append({
                    'file': str(original_file),
                    'error': "خطا در ادغام پارت‌ها"
                })
                results['success'] = [item for item in results['success'] 
                                      if Path(item['file']).name != original_file.name]
                
        except Exception as e:
            print(f"   ❌ خطای کلی در پردازش {original_file.name}: {e}")
            # ادامه به فایل بعدی
            results['failed'].append({
                'file': str(original_file),
                'error': str(e)
            })
            continue

    # پاکسازی فایل‌های PDF موقت
    print("🧹 پاکسازی فایل‌های موقت...")
    try:
        shutil.rmtree(temp_dir)
    except Exception as e:
        print(f"⚠️ خطا در حذف پوشه موقت: {e}")

    return results

# اجرای پردازش
if not pdf_files:
    print("❌ هیچ فایلی انتخاب نشده است!")
    print("💡 لطفاً به Step 1 برگردید و فایل‌ها را انتخاب کنید")
else:
    results = await split_and_process_pdfs(pdf_files)

## 📊 Step 3: گزارش نهایی

In [ ]:
# بارگذاری تابع format_time از ماژول پردازش
from pdf_extractor_processor import format_time

print("="*80)
print("📊 گزارش کامل پردازش")
print("="*80)
print()

# بررسی وجود results
if 'results' not in globals():
    print("❌ هنوز پردازشی انجام نشده است!")
    print("💡 لطفاً ابتدا Step 2 را اجرا کنید")
else:
    # فایل‌های موفق
    if results['success']:
        print(f"✅ فایل‌های موفق ({len(results['success'])}):\n")
        for item in results['success'][:20]:
            from pathlib import Path
            filename = Path(item['file']).name
            print(f"   ✓ {filename}")
            print(f"      ⏱️  {format_time(item['time'])} | 📄 {item.get('pages', 0)} صفحه | 🖼️  {item.get('images', 0)} تصویر")
        if len(results['success']) > 20:
            print(f"\n   ... و {len(results['success']) - 20} فایل دیگر")
        print()
    
    # فایل‌های ناموفق
    if results['failed']:
        print(f"❌ فایل‌های ناموفق ({len(results['failed'])}):\n")
        for item in results['failed']:
            from pathlib import Path
            filename = Path(item['file']).name
            print(f"   ✗ {filename}")
            print(f"      خطا: {item['error'][:150]}...")
            print()
    
    print("="*80)
    print(f"📁 همه خروجی‌ها در کنار فایل‌های PDF ذخیره شده‌اند")
    print("="*80)

## 🗑️ Step 4: حذف فایل‌های تولید شده

**⚠️ توجه:** این سل همه فایل‌های `.md` و پوشه‌های `images_*` تولید شده در مرحله پردازش را حذف می‌کند!

In [ ]:
# بارگذاری ماژول حذف
from pdf_extractor_cleanup import cleanup_generated_files

# اجرای حذف
if not pdf_files:
    print("❌ هیچ فایلی برای حذف یافت نشد!")
    print("? لطفاً ابتدا Step 1 را اجرا کنید")
else:
    cleanup_result = cleanup_generated_files(pdf_files)